# Membership Inference Attack (Shokri et al., 2016)

Implementación fiel del paper [Membership Inference Attacks Against Machine Learning Models](https://arxiv.org/abs/1610.05820).

## Pipeline
1. **Dataset**: 20 Newsgroups (texto, 20 clases)
2. **Split 4-way**: target_train / target_test / shadow_train / shadow_test
3. **Target model**: entrenado en target_train (con overfitting moderado)
4. **Shadow models**: N modelos que imitan al target
5. **Attack model**: clasificador binario (in/out) entrenado con probs de los shadows
6. **Métricas**: AUC, Attack Accuracy, Membership Advantage

In [ ]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (roc_auc_score, accuracy_score,
                              confusion_matrix, roc_curve,
                              precision_score, recall_score)
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

# Hiperparámetros (modificables)
N_SHADOWS = 3
TARGET_TRAIN_SIZE = 1000
SHADOW_TRAIN_SIZE = 1000
MAX_FEATURES = 3000
ATTACK_MODEL = 'random_forest'  # 'random_forest' o 'logistic'

print(f'Config: N_SHADOWS={N_SHADOWS}, TARGET_SIZE={TARGET_TRAIN_SIZE}, ATTACK={ATTACK_MODEL}')

## PASO 1 — Cargar dataset y hacer split 4-way

Necesitamos 4 conjuntos disjuntos:
- `target_train`: entrena el modelo víctima (miembros reales)
- `target_test`: no miembros reales para el ataque final
- `shadow_train`: pool para entrenar shadow models
- `shadow_test`: pool de no-miembros para shadows

In [ ]:
# Cargar 20 Newsgroups (descarga automática primera vez)
print('Descargando 20 Newsgroups...')
data = fetch_20newsgroups(
    subset='all',
    remove=('headers', 'footers', 'quotes'),
    random_state=42
)
X_all = np.array(data.data)
y_all = np.array(data.target)
n_classes = len(data.target_names)

print(f'Total documentos: {len(X_all)}')
print(f'Clases: {n_classes}')

# Shuffle
idx = np.random.permutation(len(X_all))
X_all, y_all = X_all[idx], y_all[idx]

# Split 4-way
t1 = TARGET_TRAIN_SIZE
t2 = t1 + TARGET_TRAIN_SIZE
t3 = t2 + SHADOW_TRAIN_SIZE * N_SHADOWS
t4 = t3 + SHADOW_TRAIN_SIZE * N_SHADOWS

X_target_train, y_target_train = X_all[:t1], y_all[:t1]
X_target_test,  y_target_test  = X_all[t1:t2], y_all[t1:t2]
X_shadow_pool_in,  y_shadow_pool_in  = X_all[t2:t3], y_all[t2:t3]
X_shadow_pool_out, y_shadow_pool_out = X_all[t3:t4], y_all[t3:t4]

print(f'\nSplits:')
print(f'  target_train:    {len(X_target_train)}')
print(f'  target_test:     {len(X_target_test)}')
print(f'  shadow_pool_in:  {len(X_shadow_pool_in)}')
print(f'  shadow_pool_out: {len(X_shadow_pool_out)}')

## PASO 2 — Vectorización TF-IDF

Fit en todo el corpus para que el vocabulario sea consistente entre target y shadows.

In [ ]:
vectorizer = TfidfVectorizer(
    max_features=MAX_FEATURES,
    stop_words='english',
    ngram_range=(1, 1),
    min_df=2
)
vectorizer.fit(X_all)

V_target_train     = vectorizer.transform(X_target_train)
V_target_test      = vectorizer.transform(X_target_test)
V_shadow_pool_in   = vectorizer.transform(X_shadow_pool_in)
V_shadow_pool_out  = vectorizer.transform(X_shadow_pool_out)

print(f'Vocabulario: {len(vectorizer.get_feature_names_out())}')
print(f'Shape target_train vectors: {V_target_train.shape}')

## PASO 3 — Entrenar modelo víctima (target)

Usamos `LogisticRegression` con regularización débil para forzar **overfitting moderado**.
El gap train/test es lo que el ataque va a explotar.

In [ ]:
def train_model(X, y):
    """Modelo simple con poca regularización (favorece overfitting).
    n_jobs=1 a propósito: saga multinomial con n_jobs=-1 consume mucha RAM."""
    model = LogisticRegression(
        C=10.0,           # baja regularización
        max_iter=200,
        solver='lbfgs',   # denso pero estable; bajo consumo de memoria
        n_jobs=1
    )
    model.fit(X, y)
    return model

print('Entrenando modelo target...')
target_model = train_model(V_target_train, y_target_train)

train_acc = accuracy_score(y_target_train, target_model.predict(V_target_train))
test_acc  = accuracy_score(y_target_test,  target_model.predict(V_target_test))

print(f'Target train accuracy: {train_acc:.4f}')
print(f'Target test  accuracy: {test_acc:.4f}')
print(f'Overfitting gap:       {train_acc - test_acc:.4f}  ← señal que el ataque explota')

## PASO 4 — Entrenar N shadow models y generar dataset de ataque

Cada shadow model:
1. Se entrena con un subset de `shadow_pool_in`
2. Genera vectores de probabilidad sobre sus propios miembros (label=1) y no-miembros (label=0)

In [ ]:
def get_probs(model, X, n_classes):
    """Devuelve probas con shape (n_samples, n_classes).
    Maneja modelos que no vieron todas las clases."""
    probs = model.predict_proba(X)
    full = np.zeros((X.shape[0], n_classes))
    for i, c in enumerate(model.classes_):
        full[:, c] = probs[:, i]
    return full

attack_X = []  # features = vector de probas
attack_y = []  # label = 1 (in) / 0 (out)
attack_class = []  # clase verdadera (para attack per-class)

in_chunk_size  = SHADOW_TRAIN_SIZE
out_chunk_size = SHADOW_TRAIN_SIZE

for s in range(N_SHADOWS):
    a, b = s * in_chunk_size, (s + 1) * in_chunk_size
    c, d = s * out_chunk_size, (s + 1) * out_chunk_size

    X_in,  y_in  = V_shadow_pool_in[a:b],  y_shadow_pool_in[a:b]
    X_out, y_out = V_shadow_pool_out[c:d], y_shadow_pool_out[c:d]

    shadow = train_model(X_in, y_in)

    probs_in  = get_probs(shadow, X_in,  n_classes)
    probs_out = get_probs(shadow, X_out, n_classes)

    attack_X.append(probs_in);  attack_y.extend([1]*len(probs_in));  attack_class.extend(y_in)
    attack_X.append(probs_out); attack_y.extend([0]*len(probs_out)); attack_class.extend(y_out)

    tr = accuracy_score(y_in,  shadow.predict(X_in))
    te = accuracy_score(y_out, shadow.predict(X_out))
    print(f'  Shadow {s+1}: train_acc={tr:.3f}, test_acc={te:.3f}, gap={tr-te:.3f}')

attack_X = np.vstack(attack_X)
attack_y = np.array(attack_y)
attack_class = np.array(attack_class)

print(f'\nAttack dataset: {attack_X.shape}, label balance: {attack_y.mean():.2f}')

## PASO 5 — Entrenar attack model

Clasificador binario: vector de probas → membership (0/1).
Versión simple (un solo modelo global). Variante per-class disponible más abajo.

In [ ]:
def make_attack_model():
    if ATTACK_MODEL == 'random_forest':
        return RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1)
    return LogisticRegression(max_iter=300)

attack_model = make_attack_model()
attack_model.fit(attack_X, attack_y)

print(f'Attack model entrenado: {type(attack_model).__name__}')
print(f'Train accuracy (shadow data): {attack_model.score(attack_X, attack_y):.4f}')

## PASO 6 — Atacar al modelo víctima

Pasamos `target_train` (members) y `target_test` (non-members) por el target,
y sus vectores de probas por el attack model.

In [ ]:
probs_members     = get_probs(target_model, V_target_train, n_classes)
probs_nonmembers  = get_probs(target_model, V_target_test,  n_classes)

X_eval = np.vstack([probs_members, probs_nonmembers])
y_eval = np.concatenate([np.ones(len(probs_members)), np.zeros(len(probs_nonmembers))])

scores = attack_model.predict_proba(X_eval)[:, 1]
preds  = (scores >= 0.5).astype(int)

print(f'Evaluación: {len(y_eval)} muestras ({int(y_eval.sum())} members + {int((1-y_eval).sum())} non-members)')

## PASO 7 — Métricas del ataque

In [ ]:
auc = roc_auc_score(y_eval, scores)
acc = accuracy_score(y_eval, preds)
advantage = acc - 0.5
prec = precision_score(y_eval, preds)
rec  = recall_score(y_eval, preds)
tn, fp, fn, tp = confusion_matrix(y_eval, preds).ravel()

if auc > 0.8:    risk = 'HIGH RISK'
elif auc > 0.65: risk = 'MODERATE RISK'
else:            risk = 'LOW RISK'

print('='*55)
print('  MEMBERSHIP INFERENCE — RESULTADOS')
print('='*55)
print(f'  AUC-ROC:               {auc:.4f}')
print(f'  Attack Accuracy:       {acc:.4f}')
print(f'  Membership Advantage:  {advantage:.4f}')
print(f'  Precision (members):   {prec:.4f}')
print(f'  Recall    (members):   {rec:.4f}')
print(f'  Confusion: TP={tp}, FP={fp}, TN={tn}, FN={fn}')
print(f'  Privacy Risk:          {risk}')
print('='*55)

## PASO 8 — Visualización

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ROC
fpr, tpr, _ = roc_curve(y_eval, scores)
axes[0].plot(fpr, tpr, 'b-', lw=2, label=f'Attack (AUC={auc:.3f})')
axes[0].plot([0,1],[0,1],'r--', label='Random')
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[0].set_title('ROC Curve'); axes[0].legend(); axes[0].grid(alpha=0.3)

# Score distribution
axes[1].hist(scores[y_eval==1], bins=30, alpha=0.6, label='Members', color='red')
axes[1].hist(scores[y_eval==0], bins=30, alpha=0.6, label='Non-members', color='blue')
axes[1].axvline(0.5, color='green', ls='--', label='Threshold')
axes[1].set_xlabel('Membership score'); axes[1].set_ylabel('Count')
axes[1].set_title('Score Distribution'); axes[1].legend()

# Confusion matrix
cm = confusion_matrix(y_eval, preds)
im = axes[2].imshow(cm, cmap='Blues')
axes[2].set_xticks([0,1]); axes[2].set_yticks([0,1])
axes[2].set_xticklabels(['Non-member','Member'])
axes[2].set_yticklabels(['Non-member','Member'])
axes[2].set_xlabel('Predicted'); axes[2].set_ylabel('Actual')
axes[2].set_title('Confusion Matrix')
for i in range(2):
    for j in range(2):
        axes[2].text(j, i, cm[i,j], ha='center', va='center', fontsize=14, fontweight='bold')
plt.colorbar(im, ax=axes[2])

plt.tight_layout()
plt.show()

## PASO 9 — Función reutilizable

Para correr nuevos experimentos sin reescribir todo (cambiar N_SHADOWS, modelo, regularización, etc).

In [ ]:
def run_mi_attack(X_train, y_train, X_test, y_test,
                  X_shadow_in, y_shadow_in,
                  X_shadow_out, y_shadow_out,
                  n_shadows=5, n_classes=20,
                  attack_kind='random_forest'):
    """Pipeline completo. Devuelve dict con métricas."""
    target = train_model(X_train, y_train)

    chunk_in  = len(y_shadow_in)  // n_shadows
    chunk_out = len(y_shadow_out) // n_shadows
    aX, ay = [], []
    for s in range(n_shadows):
        Xi = X_shadow_in[s*chunk_in:(s+1)*chunk_in]
        yi = y_shadow_in[s*chunk_in:(s+1)*chunk_in]
        Xo = X_shadow_out[s*chunk_out:(s+1)*chunk_out]
        sh = train_model(Xi, yi)
        aX.append(get_probs(sh, Xi, n_classes)); ay.extend([1]*Xi.shape[0])
        aX.append(get_probs(sh, Xo, n_classes)); ay.extend([0]*Xo.shape[0])
    aX, ay = np.vstack(aX), np.array(ay)

    am = (RandomForestClassifier(n_estimators=200, max_depth=10, n_jobs=-1, random_state=42)
          if attack_kind == 'random_forest' else LogisticRegression(max_iter=300))
    am.fit(aX, ay)

    pm = get_probs(target, X_train, n_classes)
    pn = get_probs(target, X_test,  n_classes)
    Xe = np.vstack([pm, pn])
    ye = np.concatenate([np.ones(len(pm)), np.zeros(len(pn))])
    sc = am.predict_proba(Xe)[:, 1]
    pr = (sc >= 0.5).astype(int)

    return {
        'auc': roc_auc_score(ye, sc),
        'accuracy': accuracy_score(ye, pr),
        'advantage': accuracy_score(ye, pr) - 0.5,
        'target_train_acc': accuracy_score(y_train, target.predict(X_train)),
        'target_test_acc':  accuracy_score(y_test,  target.predict(X_test)),
    }

# Ejemplo: re-correr con menos shadows
result = run_mi_attack(
    V_target_train, y_target_train, V_target_test, y_target_test,
    V_shadow_pool_in, y_shadow_pool_in,
    V_shadow_pool_out, y_shadow_pool_out,
    n_shadows=3, n_classes=n_classes
)
pd.DataFrame([result])